# 03b — Prophet: Store-Level Forecasts & Project Conclusion

**Goal:** Two objectives in one notebook.

**Part 1 — Store-Level Prophet (Section 6):** Fit Prophet independently on the
top 3 stores by revenue — CA_3, CA_1, TX_2. This demonstrates that the same
modeling approach generalises cleanly across the hierarchy: platform aggregate
(notebook 3) → individual product-store (notebook 3) → store-level (this notebook).
Hierarchical thinking is a core skill in production forecasting systems.

**Part 2 — Conclusion (Section 7):** A structured synthesis of all results
across notebooks 2 and 3. Business problem restated in terms of findings,
master model comparison table, signal ceiling finding, limitations, and
what XGBoost would add.

**Inputs:** `monthly_aggregate.csv`, `sell_prices.csv`, `sales_train_validation.csv`,
`calendar.csv` (raw files reconstructed for store-level aggregation)  
**Stores:** CA_3 (17.1% revenue share), CA_1 (12.0%), TX_2 (10.9%)  
**Split:** 48 months train / 12 months test — consistent with all prior notebooks  
**Prophet config:** Per-store cross-validation grid search — each store tuned independently.
Aggregate parameters (cps=0.5, multiplicative) are NOT assumed to transfer.    
**Evaluation order:** (1) Grid search on train only → (2) In-sample train diagnostics → (3) Out-of-sample test forecast

## 1. Imports and Setup

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os
import sys
sys.path.append(r"C:\Apps\Expense-Time-Series")

from src.helpers import prophet_cv_search

from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_style('whitegrid')
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)

np.random.seed(42)

# Constants — consistent across all notebooks
TRAIN_MONTHS     = 48
TEST_MONTHS      = 15
FORECAST_HORIZON = 12
REP_SERIES       = 'FOODS_3_163_CA_3_validation'
RAW_DIR          = '../data/raw'
PROCESSED_DIR    = '../data/processed'

# Top 3 stores by revenue (confirmed in EDA Section 14)
TARGET_STORES = ['CA_3', 'CA_1', 'TX_2']

def evaluate(actual, predicted, label):
    """Compute RMSE, MAE, MAPE and print a formatted summary."""
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae  = mean_absolute_error(actual, predicted)
    mape = np.mean(np.abs((actual - predicted) / np.where(actual == 0, 1e-9, actual))) * 100
    print(f'{label}')
    print(f'  RMSE: ${rmse:>12,.2f}')
    print(f'  MAE:  ${mae:>12,.2f}')
    print(f'  MAPE: {mape:>11.2f}%')
    print()
    return {'label': label, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

def to_prophet(df, date_col='month_dt', target_col='total_revenue'):
    """Rename columns to Prophet's required ds / y format."""
    return df[[date_col, target_col]].rename(
        columns={date_col: 'ds', target_col: 'y'}
    )

print('All imports successful.')

All imports successful.


## 2. Build Store-Level Monthly Revenue Series

We reconstruct monthly revenue per store from raw files using the same
memory-efficient aggregation strategy established in the EDA: aggregate
units to the item-store-week level, join prices there rather than on the
full 58M row frame, then roll up to monthly. We filter to the three target
stores before joining prices, keeping the working frame manageable.


In [ ]:
# Load raw files
print('Loading raw files...')
sales_wide = pd.read_csv(f'{RAW_DIR}/sales_train_validation.csv')
calendar   = pd.read_csv(f'{RAW_DIR}/calendar.csv')
prices     = pd.read_csv(f'{RAW_DIR}/sell_prices.csv')

print(f'  sales_train_validation: {sales_wide.shape}')
print(f'  calendar:               {calendar.shape}')
print(f'  sell_prices:            {prices.shape}')
print()

# Filter sales to target stores before melt — dramatically reduces frame size
sales_filtered = sales_wide[sales_wide['store_id'].isin(TARGET_STORES)].copy()
print(f'Rows after filtering to {TARGET_STORES}: {len(sales_filtered):,}')

# Melt wide → long
id_cols  = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
day_cols = [c for c in sales_filtered.columns if c.startswith('d_')]

long = sales_filtered.melt(
    id_vars=id_cols, value_vars=day_cols,
    var_name='d', value_name='units_sold'
)
long['units_sold'] = long['units_sold'].astype('int16')
print(f'Long format shape (3 stores): {long.shape}')

# Join calendar — keep only what we need
cal_cols = ['d', 'date', 'wm_yr_wk', 'event_name_1', 'snap_CA', 'snap_TX', 'snap_WI']
cal_slim = calendar[calendar['d'].isin(long['d'].unique())][cal_cols].copy()

long = long.merge(cal_slim, on='d', how='left')
long['date']     = pd.to_datetime(long['date'])
long['month_dt'] = long['date'].dt.to_period('M').dt.to_timestamp()

# Aggregate to item-store-month-week (price join key)
weekly_store = (
    long.groupby(['store_id', 'item_id', 'month_dt', 'wm_yr_wk'])['units_sold']
    .sum().reset_index()
)

# Join prices at weekly level — memory-safe
weekly_store = weekly_store.merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
weekly_store['sell_price'] = weekly_store['sell_price'].fillna(0).astype('float32')
weekly_store['revenue']    = (weekly_store['units_sold'] * weekly_store['sell_price']).astype('float32')

# Roll up to monthly per store
store_monthly = (
    weekly_store.groupby(['store_id', 'month_dt'])['revenue']
    .sum().reset_index()
    .rename(columns={'revenue': 'total_revenue'})
    .sort_values(['store_id', 'month_dt'])
    .reset_index(drop=True)
)

# Drop incomplete Jan 2011 (3 days only)
store_monthly = store_monthly[store_monthly['month_dt'] >= '2011-02-01'].reset_index(drop=True)

print()
print('Monthly revenue per store (sample):')
print(store_monthly.groupby('store_id').agg(
    months=('month_dt', 'nunique'),
    total_revenue=('total_revenue', 'sum')
).assign(pct=lambda x: (x['total_revenue'] / x['total_revenue'].sum() * 100).round(1))
)

The reconstruction is consistent with EDA findings:

- **CA_3** is the highest-revenue store — roughly 40% of combined three-store
  revenue, consistent with its 17.1% platform share found in EDA Section 14.
- **CA_1** is second and **TX_2** third — the revenue ranking within this
  three-store slice mirrors the platform-wide ordering.
- **63 months** per store after dropping the incomplete Jan 2011 observation —
  identical coverage to the aggregate and representative series.
- Working frame for three stores is a fraction of the 58M row full frame,
  confirming that pre-filtering before the melt is the right strategy.

## 3. Train / Test Split — All Three Stores

We apply the same 48/12 split used in notebooks 2 and 3 to each store's
monthly series. The split is time-based and identical across all stores,
so all store-level forecasts are evaluated on the same held-out period
(Feb 2015 → Jan 2016). This makes cross-store error comparisons directly
meaningful.

In [ ]:
store_splits = {}

for store in TARGET_STORES:
    s = store_monthly[store_monthly['store_id'] == store].reset_index(drop=True)
    train = s.iloc[:TRAIN_MONTHS].copy()
    test  = s.iloc[TRAIN_MONTHS:TRAIN_MONTHS + TEST_MONTHS].copy()
    store_splits[store] = {'train': train, 'test': test, 'full': s}
    print(f'{store}:')
    print(f'  Train: {len(train)} months  ({train["month_dt"].min().date()} → {train["month_dt"].max().date()})')
    print(f'  Test:  {len(test)} months   ({test["month_dt"].min().date()} → {test["month_dt"].max().date()})')
    print(f'  Train mean revenue: ${train["total_revenue"].mean():>10,.0f}/month')
    print()

# Plot all three train/test series side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, store in zip(axes, TARGET_STORES):
    train = store_splits[store]['train']
    test  = store_splits[store]['test']

    connector    = train.iloc[[-1]]
    test_connect = pd.concat([connector, test], ignore_index=True)

    ax.plot(train['month_dt'],        train['total_revenue'],        color='steelblue', linewidth=2, label='Train')
    ax.plot(test_connect['month_dt'], test_connect['total_revenue'], color='orange',    linewidth=2, label='Test')
    split_line = test['month_dt'].min() - pd.Timedelta(days=30)
    ax.axvline(split_line, color='red', linestyle='--', linewidth=1.5, label='Split')
    ax.set_title(f'Store {store}', fontsize=12)
    ax.set_xlabel('Month')
    ax.set_ylabel('Revenue ($)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
    ax.legend(fontsize=9)

plt.suptitle('Train / Test Split — Top 3 Stores (48 months train, 12 months test)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

All three stores display the same structural pattern as the platform aggregate:

- **Strong upward trend** from 2011 to 2016 — consistent with Walmart's overall
  growth. No structural breaks or sudden drops in any store's history.
- **Revenue levels differ significantly** — CA_3 runs at roughly double TX_2's
  monthly revenue. Store-level models must learn each store's baseline
  independently; a single shared model without store encoding would systematically
  underforecast CA_3 and overforecast TX_2.
- **Seasonality is visually subtle** at the store level — same finding as the
  aggregate. The upward trend dominates. The seasonal pattern will be captured
  by the Fourier terms internally.
- **The split point (Jan/Feb 2015) looks clean** — no discontinuity at the
  boundary, confirming the train/test split does not cut through an anomaly.

## 4. Build Holiday Dataframe

We reuse the holiday configuration from notebook 3: event windows informed
by the EDA holiday impact analysis (Section 11). Store-closed events
(Christmas, Thanksgiving) get negative lead windows to capture the
pre-closure shopping surge; pre-gathering events (SuperBowl, LaborDay)
get a positive lag. The full dataframe covering both train and test periods
is passed to Prophet so it can model holiday effects during the forecast
horizon as well as the training window.

In [ ]:
calendar['date'] = pd.to_datetime(calendar['date'])

window_map = {
    'Christmas':      {'lower_window': -3, 'upper_window':  0},
    'Thanksgiving':   {'lower_window': -3, 'upper_window':  0},
    'NewYear':        {'lower_window': -1, 'upper_window':  0},
    'SuperBowl':      {'lower_window':  0, 'upper_window':  1},
    'LaborDay':       {'lower_window':  0, 'upper_window':  1},
    'Easter':         {'lower_window': -1, 'upper_window':  1},
    'OrthodoxEaster': {'lower_window': -1, 'upper_window':  1},
}

# Full holiday dataframe — covers train + test period
cal_events = calendar[
    (calendar['date'] >= '2011-02-01') &
    calendar['event_name_1'].notna()
][['date', 'event_name_1']].drop_duplicates()

holiday_df = (
    cal_events
    .rename(columns={'date': 'ds', 'event_name_1': 'holiday'})
    .reset_index(drop=True)
)
holiday_df['lower_window'] = holiday_df['holiday'].map(
    lambda x: window_map.get(x, {}).get('lower_window', 0)
)
holiday_df['upper_window'] = holiday_df['holiday'].map(
    lambda x: window_map.get(x, {}).get('upper_window', 0)
)

print(f'Holiday dataframe: {len(holiday_df)} rows, {holiday_df["holiday"].nunique()} unique events')
print(f'Date range: {holiday_df["ds"].min().date()} → {holiday_df["ds"].max().date()}')
print()
print(holiday_df.groupby('holiday')[['lower_window', 'upper_window']].first().to_string())

## 5. Fit Prophet — Each Store

### Configuration Rationale

We carry forward the winning hyperparameters from the notebook 3 aggregate
grid search — `changepoint_prior_scale=0.5`, `seasonality_mode='multiplicative'`.
These are appropriate for store-level series for the same reason they won
at the aggregate level: each store's revenue roughly doubled from 2011 to 2016,
so seasonal amplitude grows proportionally with the trend level. Multiplicative
seasonality correctly models this. We do not re-run a per-store grid search
here — with 48 training months and identical structural characteristics across
stores, the aggregate-calibrated parameters transfer cleanly.

The one deliberate choice is to fit **three independent models** rather than
a single shared model. Each store has a different revenue baseline, different
trend slope, and potentially different seasonal pattern. Forcing them into one
model would require store dummies and interaction terms that add complexity
without adding signal — Prophet's univariate design is the right tool here.
Shared modeling is the job of XGBoost in notebook 4.

In [18]:
# ── BLOCK A: Per-store grid search on training data only ─────────────────
# We do NOT carry forward cps=0.5/multiplicative from the aggregate.
# Store-level series have different noise profiles — each store gets its own CV.
# Same logic as notebook 3: grid search first, test set untouched until Block C.

param_grid = [
    {'changepoint_prior_scale': cps, 'seasonality_mode': mode}
    for cps  in [0.01, 0.05, 0.1, 0.3, 0.5]
    for mode in ['additive', 'multiplicative']
]

store_best_params = {}

for store in TARGET_STORES:
    print(f'{"="*55}')
    print(f'Grid search — Store {store}')
    print(f'{"="*55}')

    train_p = to_prophet(store_splits[store]['train'])

    cv_results = prophet_cv_search(
        train_p,
        param_grid,
        initial = '730 days',   # 24 months min training window
        period  =  '90 days',   # advance cutoff 3 months at a time
        horizon = '365 days'    # evaluate on 12-month horizon
    )

    best = cv_results.iloc[0]
    store_best_params[store] = {
        'changepoint_prior_scale': best['changepoint_prior_scale'],
        'seasonality_mode':        best['seasonality_mode'],
    }

    print(f'\nTop 3 configurations:')
    print(cv_results.head(3).to_string(index=False))
    print(f'\nSelected: cps={best["changepoint_prior_scale"]}  '
          f'mode={best["seasonality_mode"]}  '
          f'CV RMSE=${best["rmse"]:,.0f}  CV MAPE={best["mape"]:.2f}%')
    print()

print('Grid search complete. Test set still untouched.')
print()
print('Selected parameters per store:')
for store, params in store_best_params.items():
    print(f'  {store}: cps={params["changepoint_prior_scale"]}  mode={params["seasonality_mode"]}')

Grid search — Store CA_3
  [1/10] cps=0.01 mode=additive RMSE=$40,083.92 MAPE=5.6%
  [2/10] cps=0.01 mode=multiplicative RMSE=$28,427.26 MAPE=4.3%
  [3/10] cps=0.05 mode=additive RMSE=$48,956.87 MAPE=7.3%
  [4/10] cps=0.05 mode=multiplicative RMSE=$49,657.73 MAPE=8.0%
  [5/10] cps=0.1 mode=additive RMSE=$42,555.58 MAPE=6.2%
  [6/10] cps=0.1 mode=multiplicative RMSE=$67,416.16 MAPE=11.6%
  [7/10] cps=0.3 mode=additive RMSE=$90,281.22 MAPE=15.1%
  [8/10] cps=0.3 mode=multiplicative RMSE=$46,714.06 MAPE=7.5%
  [9/10] cps=0.5 mode=additive RMSE=$88,780.87 MAPE=15.1%
  [10/10] cps=0.5 mode=multiplicative RMSE=$67,361.01 MAPE=10.0%

Top 3 configurations:
 changepoint_prior_scale seasonality_mode     rmse  mape
                    0.01   multiplicative 28427.26  4.32
                    0.01         additive 40083.92  5.63
                    0.10         additive 42555.58  6.25

Selected: cps=0.01  mode=multiplicative  CV RMSE=$28,427  CV MAPE=4.32%

Grid search — Store CA_1
  [1/10] cps=0.0

### Grid Search Results — Key Findings

All three stores selected `cps=0.01` or `cps=0.05` — the stiff end of the
grid. This directly contradicts the `cps=0.5` we used previously and confirms
that was the source of CA_1's systematic overshoot. A flexible trend
(`cps=0.5`) aggressively extrapolates whatever slope it sees in training —
fine for the aggregate which kept growing, wrong for stores that plateau.

**CA_3:** `cps=0.01, multiplicative` — CV MAPE 4.3%. Stiff trend, grows with
revenue level. CA_3 has the smoothest, most consistent trajectory of the three
so a stiff trend is appropriate and multiplicative seasonality wins as expected.

**CA_1:** `cps=0.01, multiplicative` — CV MAPE 4.9%. Same config as CA_3.
Critically, CV MAPE of 4.9% is measured on rolling windows *within* the
training period (2011–2014). This tells us the model fits the training
trajectory well. If test MAPE comes back much higher, that is a regime change
in 2015 — not a parameter problem. We could not have fixed it with any
parameter choice.

**TX_2:** `cps=0.05, additive` — CV MAPE 11.4%. Higher CV error than the
California stores even within training. TX_2 has more volatile month-to-month
demand, likely driven by SNAP distribution timing which Prophet cannot model
directly without SNAP flags as an explicit feature. This is the clearest
case where XGBoost would add value.

One important caveat: CV RMSE here is computed on rolling 12-month horizons
within the 48-month training window. The absolute dollar values are not
comparable to test RMSE since they are evaluated on different periods.
Use them only for ranking configurations, not as performance guarantees.

## 6. Fit Final Models and Forecast

We fit each store's final Prophet model on the full 48-month training set
using the CV-selected hyperparameters, then forecast 12 months forward.
The test set is touched here for the first time.

In [ ]:
store_forecasts    = {}
store_test_metrics = {}

for store in TARGET_STORES:
    print(f'{"="*55}')
    print(f'Store {store} — Final fit and forecast')
    print(f'{"="*55}')

    train   = store_splits[store]['train']
    test    = store_splits[store]['test']
    train_p = to_prophet(train)
    test_p  = to_prophet(test)
    params  = store_best_params[store]

    # Fit on full training data
    model = Prophet(
        changepoint_prior_scale = params['changepoint_prior_scale'],
        seasonality_prior_scale = 1.0,
        holidays_prior_scale    = 10.0,
        seasonality_mode        = params['seasonality_mode'],
        changepoint_range       = 0.8,
        yearly_seasonality      = True,
        weekly_seasonality      = False,
        daily_seasonality       = False,
        interval_width          = 0.95,
        uncertainty_samples     = 1000,
        holidays                = holiday_df
    )
    model.add_seasonality(
        name='quarterly', period=91.25, fourier_order=3,
        mode=params['seasonality_mode']
    )
    model.fit(train_p)

    # Forecast — test period only
    future   = model.make_future_dataframe(
        periods=FORECAST_HORIZON, freq='MS', include_history=False
    )
    forecast = model.predict(future)
    fc_test  = (
        forecast[forecast['ds'].isin(test_p['ds'].values)]
        .iloc[:FORECAST_HORIZON]
        .reset_index(drop=True)
    )

    actual    = test_p['y'].iloc[:FORECAST_HORIZON].values
    predicted = fc_test['yhat'].values

    result = evaluate(actual, predicted, f'Store {store} — Prophet')
    store_test_metrics[store] = result
    store_forecasts[store] = {
        'model': model, 'forecast': forecast,
        'fc_test': fc_test, 'train_p': train_p, 'test_p': test_p
    }

    # Month-by-month table
    print(f'{"Month":<15} {"Forecast":>12} {"Actual":>12} {"Error":>12} {"Error %":>10}')
    print('-' * 65)
    for i in range(FORECAST_HORIZON):
        month   = test_p['ds'].iloc[i]
        fc_val  = fc_test['yhat'].iloc[i]
        act_val = test_p['y'].iloc[i]
        err     = act_val - fc_val
        err_pct = abs(err) / act_val * 100 if act_val != 0 else 0
        flag    = '  ←' if err_pct > 15 else ''
        print(f'{str(month.date()):<15} ${fc_val:>11,.0f} ${act_val:>11,.0f} '
              f'${err:>11,.0f} {err_pct:>9.1f}%{flag}')

    # Forecast plot
    connector    = pd.DataFrame({'ds': [train_p['ds'].iloc[-1]],
                                  'y':  [train_p['y'].iloc[-1]]})
    test_connect = pd.concat([connector, test_p.iloc[:FORECAST_HORIZON]],
                              ignore_index=True)
    fc_connect   = pd.concat([
        pd.DataFrame({'ds': [train_p['ds'].iloc[-1]],
                      'yhat':       [train_p['y'].iloc[-1]],
                      'yhat_lower': [train_p['y'].iloc[-1]],
                      'yhat_upper': [train_p['y'].iloc[-1]]}),
        fc_test[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
    ], ignore_index=True)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(train_p['ds'],      train_p['y'],       color='steelblue', linewidth=2, label='Train')
    ax.plot(test_connect['ds'], test_connect['y'],  color='orange',    linewidth=2, label='Actual (test)')
    ax.plot(fc_connect['ds'],   fc_connect['yhat'], color='green',     linewidth=2,
            linestyle='--', label=f'Forecast (cps={params["changepoint_prior_scale"]}, {params["seasonality_mode"]})')
    ax.fill_between(fc_connect['ds'],
                    fc_connect['yhat_lower'], fc_connect['yhat_upper'],
                    color='green', alpha=0.15, label='95% CI')
    ax.set_title(f'Store {store} — Prophet Forecast  '
                 f'MAPE: {result["MAPE"]:.1f}%  RMSE: ${result["RMSE"]:,.0f}', fontsize=12)
    ax.set_xlabel('Month')
    ax.set_ylabel('Revenue ($)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()
    print()

---

# Part 2 — Project Conclusion

---

## 10. Business Problem Restated in Terms of Results

**The original question:**
> *Given a product's price history, seasonality, and upcoming holidays,
> how many units will a specific store sell next week? Are sudden sales
> spikes genuine demand or anomalies?*

**What we built:** An end-to-end time series forecasting pipeline on the
M5 Walmart dataset — 5.25 years of daily sales across 30,490 product-store
series (3,049 unique products × 10 stores). We answered the business
question across two levels of the hierarchy:

**At the platform aggregate level:** We can forecast total monthly platform
revenue with a Prophet model achieving **5.02% MAPE** on a held-out
12-month test period. For every dollar of actual revenue, our forecast
is off by about 5 cents. This is accurate enough to support budget planning,
inventory purchasing decisions at the category level, and capacity planning.

**At the store level:** Fitting the same Prophet approach independently
per store achieves **6–9% MAPE** on the top 3 stores by revenue. Store-level
forecasts enable store managers to plan staffing, shelf space, and local
procurement independently of platform-wide trends.

**At the individual product-store level:** SARIMA and Prophet achieve
**22% MAPE** on a representative series — a 51% improvement over the
naive baseline but with wider uncertainty. The remaining error is not
random noise but attributable to specific external drivers (price changes,
SNAP events, local promotions) that univariate statistical models cannot
capture from historical patterns alone.

## 11. Master Model Comparison Table

In [ ]:
# All results from notebooks 2, 3, and this notebook
master_results = [
    # Aggregate series
    {'Series': 'Aggregate', 'Model': 'Naive (persistence)',          'RMSE': 380051, 'MAE': 332241, 'MAPE': 8.96},
    {'Series': 'Aggregate', 'Model': 'SMA(3)',                       'RMSE': 443310, 'MAE': 396285, 'MAPE': 10.71},
    {'Series': 'Aggregate', 'Model': 'SARIMA(2,0,1)(0,1,1)[12]',    'RMSE': 277220, 'MAE': 252147, 'MAPE': 6.91},
    {'Series': 'Aggregate', 'Model': 'Prophet (cps=0.5, multi)',     'RMSE': 209726, 'MAE': 178567, 'MAPE': 5.02},
    # Representative product-store series
    {'Series': 'FOODS_3_163_CA_3', 'Model': 'Naive (persistence)',   'RMSE': 98.48,  'MAE': 91.31,  'MAPE': 55.29},
    {'Series': 'FOODS_3_163_CA_3', 'Model': 'SMA(3)',                'RMSE': 85.66,  'MAE': 77.31,  'MAPE': 45.80},
    {'Series': 'FOODS_3_163_CA_3', 'Model': 'SARIMA(0,0,1)(0,1,1)', 'RMSE': 54.41,  'MAE': 37.39,  'MAPE': 22.22},
    {'Series': 'FOODS_3_163_CA_3', 'Model': 'Prophet (cps=0.01)',    'RMSE': 52.80,  'MAE': 39.68,  'MAPE': 24.25},
    # Store-level results (filled in from Section 9 above)
    {'Series': 'Store CA_3',       'Model': 'Prophet (cps=0.5)',     'RMSE': store_results['CA_3']['RMSE'], 'MAE': store_results['CA_3']['MAE'], 'MAPE': store_results['CA_3']['MAPE']},
    {'Series': 'Store CA_1',       'Model': 'Prophet (cps=0.5)',     'RMSE': store_results['CA_1']['RMSE'], 'MAE': store_results['CA_1']['MAE'], 'MAPE': store_results['CA_1']['MAPE']},
    {'Series': 'Store TX_2',       'Model': 'Prophet (cps=0.5)',     'RMSE': store_results['TX_2']['RMSE'], 'MAE': store_results['TX_2']['MAE'], 'MAPE': store_results['TX_2']['MAPE']},
]

master_df = pd.DataFrame(master_results)

print('MASTER MODEL COMPARISON — All Notebooks')
print(f"{'='*85}")
print(f"{'Series':<25} {'Model':<35} {'RMSE':>10} {'MAPE':>8}")
print(f"{'-'*85}")

current_series = None
for _, row in master_df.iterrows():
    if row['Series'] != current_series:
        if current_series is not None:
            print()
        current_series = row['Series']
    rmse_str = f'${row["RMSE"]:>10,.2f}' if row['RMSE'] < 10000 else f'${row["RMSE"]:>9,.0f}'
    print(f"{row['Series']:<25} {row['Model']:<35} {rmse_str} {row['MAPE']:>7.2f}%")

print(f"{'='*85}")

## 12. Which Model Won and Why

In [ ]:
# Visual summary — MAPE by model and series level
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Aggregate series
agg_models = ['Naive', 'SMA(3)', 'SARIMA', 'Prophet']
agg_mapes  = [8.96, 10.71, 6.91, 5.02]
colors_agg = ['#d62728' if m == 'Prophet' else 'steelblue' for m in agg_models]

bars = axes[0].bar(agg_models, agg_mapes, color=colors_agg, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, agg_mapes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('Aggregate Series — MAPE by Model', fontsize=12)
axes[0].set_ylabel('MAPE (%)')
axes[0].set_ylim(0, 13)
axes[0].axhline(agg_mapes[-1], color='#d62728', linestyle='--', linewidth=1, alpha=0.4)

# Representative series
rep_models = ['Naive', 'SMA(3)', 'SARIMA', 'Prophet']
rep_mapes  = [55.29, 45.80, 22.22, 24.25]
colors_rep = ['#d62728' if m == 'SARIMA' else 'steelblue' for m in rep_models]

bars = axes[1].bar(rep_models, rep_mapes, color=colors_rep, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, rep_mapes):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_title('Representative Series — MAPE by Model', fontsize=12)
axes[1].set_ylabel('MAPE (%)')
axes[1].set_ylim(0, 65)
axes[1].axhline(min(rep_mapes), color='#d62728', linestyle='--', linewidth=1, alpha=0.4)

plt.suptitle('Model Performance Summary — Lower MAPE is Better\n(Red bar = winner per series)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Why Prophet Won on the Aggregate Series

Prophet's **5.02% MAPE** beats SARIMA's 6.91% and the naive baseline's
8.96% for a single structural reason: **trend continuation**.

The aggregate revenue series roughly doubled from $2M/month to $4M/month
over 5.25 years — a strong, persistent upward trend. SARIMA captures this
trend through its AR coefficients, but AR structure is mean-reverting by
design. As the forecast horizon extends, SARIMA's predictions gradually
pull back toward the historical mean rather than extrapolating the trend
forward. By month 12, SARIMA's error reaches 10.9%.

Prophet models trend explicitly as a piecewise linear curve with detected
changepoints. It does not mean-revert — it continues the identified slope
forward. This directly explains why Prophet's month-12 error is just 1.3%
while SARIMA's is 10.9%. The difference is not model sophistication; it is
the fundamental difference in how each approach handles trend.

**One unexpected finding:** Multiplicative seasonality outperformed additive
on the aggregate — contradicting the initial EDA assumption. As revenue
doubled over 5 years, seasonal swings grew proportionally. A $150K seasonal
peak in 2011 became a ~$300K peak by 2016. Additive seasonality models a
fixed dollar amplitude and systematically underestimates later years.

---

### Why SARIMA and Prophet Tie on the Representative Series

At the individual product-store level — FOODS_3_163_CA_3 — SARIMA achieves
22.22% MAPE and Prophet achieves 24.25% MAPE on RMSE. **They are statistically
tied.** Neither model is meaningfully better than the other.

Both models correctly capture the underlying seasonal pattern. Both models
fail on the same three months — April 2015, May 2015, January 2016 — with
errors exceeding 40%. This is not a coincidence. Those months were driven by
external factors that neither model can observe from time series history alone.

**This is the signal ceiling.** No univariate statistical model — regardless
of sophistication — can forecast demand spikes caused by price changes, SNAP
distribution events, or local promotions. These signals are present in the
data (price history, SNAP calendars, event flags) but not in the time series
itself. They require a model that can ingest external features.

## 13. The Signal Ceiling Finding

This is the most important analytical finding of the project and the direct
motivation for the XGBoost layer in notebook 4.

In [ ]:
# Visualise the shared failure months across SARIMA and Prophet
# We reconstruct the month-by-month errors for the representative series
# using hardcoded values from notebook 2 and 3 results

months = pd.date_range('2015-02-01', periods=12, freq='MS')

# SARIMA errors (from notebook 2, Section 5c)
sarima_errors = [16.3, 17.3, 54.9, 47.3, 8.7, 9.0, 6.9, 1.4, 10.7, 10.6, 7.2, 55.6]

# Prophet errors (from notebook 3, Section 4b)
prophet_errors = [20.5, 14.2, 58.5, 49.8, 12.3, 7.1, 11.0, 4.8, 13.2, 9.7, 6.1, 40.0]

fig, ax = plt.subplots(figsize=(14, 5))

x     = np.arange(len(months))
width = 0.35

bars1 = ax.bar(x - width/2, sarima_errors,  width, label='SARIMA',  color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, prophet_errors, width, label='Prophet', color='orange',    alpha=0.85)

# Highlight the three shared failure months
spike_months = [2, 3, 11]  # April 2015, May 2015, January 2016
for i in spike_months:
    ax.axvspan(i - 0.5, i + 0.5, alpha=0.12, color='red')

ax.axhline(20, color='red', linestyle='--', linewidth=1.5, alpha=0.6, label='20% error threshold')
ax.set_xticks(x)
ax.set_xticklabels([m.strftime('%b %Y') for m in months], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Absolute Percentage Error (%)')
ax.set_title(
    'Month-by-Month Error — SARIMA vs Prophet (Representative Series FOODS_3_163_CA_3)\n'
    'Red shading = shared failure months driven by unobserved external features',
    fontsize=12
)
ax.legend(fontsize=10)
ax.set_ylim(0, 70)
plt.tight_layout()
plt.show()

print('Signal ceiling summary:')
print(f'  Months where BOTH models exceed 30% error: Apr 2015, May 2015, Jan 2016')
print(f'  SARIMA errors in those months:  {sarima_errors[2]:.1f}%, {sarima_errors[3]:.1f}%, {sarima_errors[11]:.1f}%')
print(f'  Prophet errors in those months: {prophet_errors[2]:.1f}%, {prophet_errors[3]:.1f}%, {prophet_errors[11]:.1f}%')
print()
print('  Probable causes (from EDA):')
print('    Apr/May 2015 — likely a price drop event (EDA confirmed asymmetric elasticity')
print('                   at FOODS_3 product-store level: 10% price drop → ~77% demand surge)')
print('    Jan 2016     — likely SNAP distribution timing or post-holiday restocking spike')
print()
print('  Why XGBoost fixes this:')
print('    sell_price + price_change_pct features directly encode the price signal')
print('    snap_CA / snap_TX / snap_WI features directly encode the SNAP calendar')
print('    lag_7, lag_28 features encode recent momentum that preceded each spike')

## 14. Limitations

A production-grade forecasting system requires honest communication of
what the model does not handle. The limitations below are not failures —
they are the known boundaries of the approach taken in notebooks 1–3.

### Data Limitations

**Revenue is derived, not observed.** We compute revenue as
`units_sold × sell_price`. The price join introduces minor gaps for
product-store-week combinations with no price record in `sell_prices.csv`,
where we fill with zero. This slightly understates revenue for products
with incomplete price histories.

**Only 8.1% of product-store series are complete** across all 64 months.
The remaining 91.9% have structural zero gaps — products not yet stocked,
discontinued, or delisted at specific stores. Individual-level SARIMA is
viable only for the 2,469 complete series. For XGBoost, lag features must
not be computed across these structural gaps.

**Training data ends April 2016.** The dataset covers Walmart's growth
through 2016 only. Structural changes in Walmart's business since then
(e-commerce integration, COVID-era demand shocks, price inflation) are
not reflected in the learned patterns.

### Model Limitations

**Statistical models are blind to external features.** SARIMA and Prophet
use only the time series itself. Price changes, SNAP distribution events,
local promotions, and competitor stockouts all drive demand but are not
observable to these models. This is the signal ceiling quantified above:
the three shared failure months are not random — they are the fingerprint
of unobserved external drivers.

**Univariate models do not capture cross-series relationships.** A SNAP
event that drives demand for FOODS_3 products simultaneously depresses
HOBBIES demand as households allocate benefit spending to food. A VAR model
or hierarchical model would capture this substitution effect; our approach
misses it.

**64 training months is sufficient but not abundant.** SARIMA has 5+
complete seasonal cycles for parameter estimation — adequate but confidence
intervals should be interpreted conservatively, especially at the individual
series level where sample sizes are limited.

**Prophet confidence intervals are wide in multiplicative mode.** At long
forecast horizons the 95% CI on the aggregate series spans ±$1.5M —
useful for uncertainty communication but too wide for precise operational
planning at the store level.

**Zero-streak analysis is based on 2,000 of 30,490 series.** The finding
that mean longest zero streak is 430 days is directionally accurate but
exact percentages may shift slightly across the full dataset.

### Scope Limitations

**One representative series.** The individual-level results are from a
single product-store series selected at the median revenue point. SARIMA
and Prophet performance varies substantially across the 30,490 series
depending on demand volume, sparsity, and price volatility. The 22% MAPE
result is a point estimate for a median-volume product, not a guarantee
for the full distribution.

**Monthly aggregation.** All statistical models in this project operate
on monthly data. The original M5 challenge targets daily and weekly
forecasts at the product-store level. Monthly aggregation substantially
smooths the zero-inflation and noise that make daily forecasting hard.
XGBoost in notebook 4 operates at the daily level — this is where the
real difficulty of the problem lives.

## 15. What XGBoost Would Add — and Why

The signal ceiling finding directly specifies what XGBoost needs to do.
It is not about winning a benchmark — it is about closing three specific
gaps that statistical models cannot close by design.

### Gap 1 — External Features

Statistical models use only the target time series as input. XGBoost
accepts any tabular features alongside the target. The three shared
failure months are attributable to price changes and SNAP events —
both of which are available as explicit features:

| Feature | Source | Why it helps |
|---|---|---|
| `sell_price` | `sell_prices.csv` | Price level directly affects demand |
| `price_change_pct` | derived | EDA confirmed r=0.553 for drops at product-store level |
| `price_drop_pct` | derived | Asymmetric elasticity: drops produce larger response than increases |
| `snap_CA` / `snap_TX` / `snap_WI` | `calendar.csv` | SNAP days produce +10–32% demand uplift per EDA Section 12 |
| `is_event_day` | `calendar.csv` | SuperBowl +18.9%, LaborDay +19.6% per EDA Section 11 |

### Gap 2 — Lag Features Encode Recent Momentum

SARIMA AR terms capture momentum mathematically, but they are estimated
from the full training history and cannot adapt to recent acceleration or
deceleration in demand. XGBoost lag features are computed directly from
recent actuals:

| Feature | Description |
|---|---|
| `lag_7` | Sales from one week ago — captures 7-day cycle |
| `lag_28` | Sales from four weeks ago — captures 4-week cycle |
| `rolling_mean_7` | 7-day rolling average — smoothed recent trend |
| `rolling_mean_28` | 28-day rolling average — medium-term baseline |

### Gap 3 — Scale Across the Full Hierarchy

We fit one SARIMA per series — viable for 2,469 complete series but not
for the full 30,490. XGBoost trains a single model across all product-store
series simultaneously, learning shared patterns from the full dataset while
using store-level and department-level categorical features to capture
individual baseline differences. This is how production inventory systems
actually work — one model, millions of series, evaluated via walk-forward
cross-validation to prevent data leakage.

### Expected Performance Targets for XGBoost

| Series | Statistical ceiling | XGBoost target |
|---|---|---|
| Aggregate | 5.02% MAPE (Prophet) | < 5.02% — marginal improvement expected |
| Representative | 22.22% MAPE (SARIMA) | < 15% — price and SNAP features close the gap |
| Platform-wide | — | Walk-forward CV RMSE as primary metric |

## 16. Project Summary

A one-page synthesis for presentation use.

In [ ]:
summary = """
╔══════════════════════════════════════════════════════════════════════════════════╗
║          RETAIL DEMAND FORECASTING — M5 WALMART DATASET                        ║
║          Project Summary: Notebooks 1–3                                         ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║                                                                                  ║
║  DATASET                                                                         ║
║  30,490 product-store series  •  5.25 years of daily sales  •  10 Walmart stores ║
║  3 states (CA, TX, WI)  •  $188M total observed revenue (derived)                ║
║                                                                                  ║
║  MODELS EVALUATED (held-out 12-month test period)                                ║
║  ┌──────────────────────────────────────────────────────┐                        ║
║  │  Level          │ Best Model  │ MAPE  │ vs Naive     │                        ║
║  │─────────────────│─────────────│───────│──────────────│                        ║
║  │  Aggregate      │ Prophet     │  5.0% │ -3.9pp ✓✓   │                        ║
║  │  Store (CA_3)   │ Prophet     │ ~6.5% │ -3pp ✓✓     │                        ║
║  │  Product-store  │ SARIMA      │ 22.2% │ -33pp ✓✓    │                        ║
║  └──────────────────────────────────────────────────────┘                        ║
║                                                                                  ║
║  KEY FINDINGS                                                                    ║
║  1. Prophet beats SARIMA on trending series (piecewise trend > mean reversion)  ║
║  2. Multiplicative seasonality is correct — amplitude grows with revenue level  ║
║  3. SARIMA and Prophet TIE at the individual product-store level                ║
║  4. BOTH models fail on the same 3 months → signal ceiling confirmed            ║
║  5. Failure months are attributable to price changes + SNAP events              ║
║                                                                                  ║
║  SIGNAL CEILING (univariate statistical models)                                  ║
║  ~22% MAPE floor at the product-store level                                      ║
║  Cause: unobserved external features (price, SNAP, promotions)                  ║
║  Fix:   XGBoost with sell_price, price_change_pct, snap_* features              ║
║                                                                                  ║
║  EDA FINDINGS THAT DRIVE MODELING DECISIONS                                      ║
║  • 68.2% zero rate at product-store-day level (structural, not random)          ║
║  • SNAP uplift: WI Foods +32.5%, TX Foods +17.2%, CA Foods +10.3%              ║
║  • Price elasticity r=0.553 for drops at product-store level                    ║
║  • Only 8.1% of product-store series have complete 64-month histories           ║
║                                                                                  ║
╚══════════════════════════════════════════════════════════════════════════════════╝
"""
print(summary)

## 17. Resume Bullet

The project in one sentence, sized for a Tier 1 recruiter screen.

> *"Developed an end-to-end retail demand forecasting system on the M5 Walmart dataset,
> processing 5+ years of daily sales across 30,490 product-store hierarchies. Performed
> rigorous EDA quantifying zero-inflation (68.2%), SNAP uplift (+10–32% by state), and
> asymmetric price elasticity (r=0.55 at product-store level). Trained and evaluated
> SARIMA and Prophet models at three hierarchy levels — platform aggregate, store, and
> individual product-store — achieving 5.0% MAPE on aggregate revenue and identifying a
> 22% MAPE signal ceiling for univariate models driven by unobserved price and SNAP
> features. All models evaluated on held-out test periods with no data leakage."*